# Run PRA-AOG Lite end to end

Every executable cell in this notebook is a shell command beginning with `!`.

The notebook supports two starting points:

1. **Existing terminal caches:** set `TRAIN_CACHE` and `VAL_CACHE`, then skip the cache-generation cell.
2. **A Stage-1 checkpoint:** set `PARTIMAGENET_ROOT` and `STAGE1_CKPT`, then run the cache-generation cell.

GPU is strongly recommended for cache generation and training. The default commands implement the revised primary experiment: class-agnostic terminal evidence, shared relation motifs, posterior logits, and top-K posterior parsing.

## 1. Edit paths once

Edit only the path values in the next cell. The defaults target this local checkout. Use absolute paths if you move data or caches.

Expected PartImageNet layout:

```text
PARTIMAGENET_ROOT/
  annotations/train/train.json
  annotations/val/val.json
  images/train/
  images/val/
```

When terminal caches already exist elsewhere, replace `TRAIN_CACHE` and `VAL_CACHE` with those paths.

In [ ]:
!python -c 'from pathlib import Path; p=Path("/tmp/pra_aog_notebook.env"); p.write_text("export WORKSPACE=\"/home/dfli/instance_slot_aog\"\nexport REPO_DIR=\"$WORKSPACE/clean_v18_v39_v42\"\nexport PARTIMAGENET_ROOT=\"$WORKSPACE/full_hyco/PartImageNet\"\nexport STAGE1_CKPT=\"$REPO_DIR/runs/stage1_quality_upgrade/checkpoints/stage1_best.pt\"\nexport CONFIG=\"$REPO_DIR/configs/notebook_stage1.yaml\"\nexport CACHE_DIR=\"$REPO_DIR/artifacts/strict_aog\"\nexport TRAIN_CACHE=\"$CACHE_DIR/train_strict_aog_terminals.pt\"\nexport VAL_CACHE=\"$CACHE_DIR/val_strict_aog_terminals.pt\"\nexport BUNDLE=\"$REPO_DIR/artifacts/pra_aog/pra_aog_bundle.pt\"\nexport RUN_DIR=\"$REPO_DIR/runs/pra_aog\"\nexport BEST_CKPT=\"$RUN_DIR/checkpoints/strict_aog_best.pt\"\n", encoding="utf-8"); print(p.read_text())'

## 2. Verify local checkout

In [ ]:
! . /tmp/pra_aog_notebook.env && git -C "$REPO_DIR" status --short --branch && git -C "$REPO_DIR" log -1 --oneline

## 3. Install dependencies

The `vision` extra is needed only when generating terminal caches from Stage 1, but installing it here keeps the notebook end-to-end.

In [ ]:
!python -m pip install --upgrade pip setuptools wheel

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python -m pip install -e ".[dev,vision]"

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python -c 'import torch; from partcat_hkg.pra_aog import PRAAOGParser, load_pra_aog; print("torch", torch.__version__); print("cuda_available", torch.cuda.is_available()); print("PRA-AOG import OK")' && (nvidia-smi || true)

## 4. Run unit tests

Run this before a long experiment. It checks the new posterior parser, motif sharing, visibility/readout logic, top-down verifier, and the existing strict-AOG core.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && pytest -q tests/test_pra_aog.py tests/test_strict_aog_core.py

## 5. Create a dataset-specific Stage-1 config

This writes a small YAML override and leaves the repository's original configs untouched.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python -c 'from pathlib import Path; import os; p=Path(os.environ["CONFIG"]); p.write_text("extends: stage1_quality_upgrade.yaml\npaths:\n  partimagenet_root: " + repr(os.environ["PARTIMAGENET_ROOT"]) + "\n", encoding="utf-8"); print(p.read_text())'

In [ ]:
! . /tmp/pra_aog_notebook.env && for p in "$PARTIMAGENET_ROOT/annotations/train/train.json" "$PARTIMAGENET_ROOT/annotations/val/val.json" "$PARTIMAGENET_ROOT/images/train" "$PARTIMAGENET_ROOT/images/val"; do if [ -e "$p" ]; then echo "FOUND   $p"; else echo "MISSING $p"; fi; done

## 6. Generate terminal caches Ã¢â‚¬â€ skip when caches already exist

The command is conditional: it does nothing when both cache manifests already exist.

The revised extraction settings treat object support as uncertain evidence (`post`) rather than a hard pre-mask, retain the best support component bookkeeping, shard caches, and store validation images for diagnostics.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && if [ -f "$TRAIN_CACHE" ] && [ -f "$VAL_CACHE" ]; then echo "Terminal caches already exist; skipping generation."; else test -f "$STAGE1_CKPT" || { echo "Missing Stage-1 checkpoint: $STAGE1_CKPT"; exit 1; }; mkdir -p "$CACHE_DIR"; python scripts/cache_strict_aog_terminals.py --config "$CONFIG" --stage1-ckpt "$STAGE1_CKPT" --out-dir "$CACHE_DIR" --device auto --splits train,val --batch-size 16 --num-workers 2 --threshold 0.40 --support-gate-mode post --support-component-mode best --max-terminals 32 --mask-size 64 --shard-size 1024 --store-images --store-images-splits val; fi

In [ ]:
! . /tmp/pra_aog_notebook.env && test -f "$TRAIN_CACHE" && test -f "$VAL_CACHE" && ls -lh "$TRAIN_CACHE" "$VAL_CACHE" && du -sh "$CACHE_DIR" 

## 7. Build the PartÃ¢â‚¬â€œMotifÃ¢â‚¬â€œObject PRA-AOG bundle

The primary path sets build-time role filtering to zero and uses conservative motif sharing with reuse, information-gain, MDL, and geometric-coherence controls.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && mkdir -p "$(dirname "$BUNDLE")" && python scripts/build_pra_aog.py --cache "$TRAIN_CACHE" --out "$BUNDLE" --num-templates-per-class 3 --max-slots-per-template 14 --max-slots-per-part 4 --min-template-support 2 --min-slot-support 0.12 --required-tau 0.45 --min-role-overlap 0.0 --min-edge-support 0.06 --min-edge-count 2 --min-edge-information-gain 0.02 --max-edges-per-template 24 --relation-var-floor 0.006 --geom-var-floor 0.004 --count-max 6 --motif-min-references 2 --motif-min-utility 0.0 --motif-mdl-penalty 0.01 --motif-max-standardized-distance 2.5 --motif-heterogeneity-penalty 0.05 --motif-shrinkage 0.35

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python -c 'import os; from partcat_hkg.pra_aog import load_pra_aog; b=load_pra_aog(os.environ["BUNDLE"]); g=b.grammar; print("classes", g.num_classes); print("templates_per_class", g.num_templates); print("max_slots", g.max_slots); print("edges", int(g.edges.shape[0])); print("shared_motifs", len(b.motif_bank.motifs)); print("motif_reuse_ratio", b.motif_bank.reuse_ratio); print("metadata", b.metadata)'

## 8. One-minute smoke training

Run this first. It uses only two train and two validation batches and writes into `runs/pra_aog_smoke`.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python scripts/train_pra_aog.py --bundle "$BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "${RUN_DIR}_smoke" --device auto --batch-size 4 --epochs 1 --lr 2e-4 --weight-decay 1e-4 --assignment gpu_mf --relation-weight 1.25 --count-weight 0.15 --missing-weight 0.35 --edge-start-epoch 1 --top-k 5 --posterior-tau 0.75 --posterior-logits --num-workers 0 --max-train-batches 2 --max-val-batches 2

## 9. Full training

This is the recommended primary experiment. `--preload-cache` is fastest when RAM is sufficient. Remove that flag when the cache is too large for memory. With preloading, `--num-workers 0` avoids duplicating cache memory.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python scripts/train_pra_aog.py --bundle "$BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "$RUN_DIR" --device auto --batch-size 16 --epochs 20 --lr 2e-4 --weight-decay 1e-4 --assignment gpu_mf --relation-weight 1.25 --count-weight 0.15 --missing-weight 0.35 --edge-start-epoch 1 --top-k 5 --posterior-tau 0.75 --posterior-logits --preload-cache --num-workers 0

## 10. Evaluate the saved best checkpoint

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && test -f "$BEST_CKPT" && python scripts/train_pra_aog.py --bundle "$BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "$RUN_DIR" --checkpoint "$BEST_CKPT" --eval-only --device auto --batch-size 16 --assignment gpu_mf --relation-weight 1.25 --count-weight 0.15 --missing-weight 0.35 --top-k 5 --posterior-tau 0.75 --posterior-logits --preload-cache --num-workers 0

## 11. Decode one validation image

This saves a JSON summary containing the calibrated class posterior, top-K parse forest, slot visibility states, relations, counts, localization, uncertainty, and bounded top-down queries. Posterior semantic masks are saved separately as a PyTorch tensor.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python scripts/infer_pra_aog.py --bundle "$BUNDLE" --cache "$VAL_CACHE" --checkpoint "$BEST_CKPT" --out-dir "$RUN_DIR/inference" --sample-index 0 --device auto --assignment gpu_mf --top-k 5 --posterior-tau 0.75 --relation-weight 1.25 --count-weight 0.15 --missing-weight 0.35 --posterior-logits

In [ ]:
! . /tmp/pra_aog_notebook.env && python -m json.tool "$RUN_DIR/inference/sample_00000_summary.json" | head -n 220

## 12. Inspect and package outputs

In [ ]:
! . /tmp/pra_aog_notebook.env && find "$RUN_DIR" -maxdepth 4 -type f | sort

In [ ]:
! . /tmp/pra_aog_notebook.env && tar -czf "$WORKSPACE/pra_aog_results.tar.gz" -C "$(dirname "$RUN_DIR")" "$(basename "$RUN_DIR")" && ls -lh "$WORKSPACE/pra_aog_results.tar.gz" 

## Optional ablation: expose class-conditioned role evidence

Do **not** use this for the primary result. It is included only to measure how much candidate-class Stage-1 role maps affect performance.

Add `--use-class-role-evidence` to both training and evaluation commands and use a separate output directory.

In [ ]:
! . /tmp/pra_aog_notebook.env && cd "$REPO_DIR" && python scripts/train_pra_aog.py --bundle "$BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "${RUN_DIR}_role_ablation" --device auto --batch-size 16 --epochs 20 --assignment gpu_mf --top-k 5 --posterior-tau 0.75 --posterior-logits --use-class-role-evidence --preload-cache --num-workers 0